# 6. Loop dmr motif

Part of the **[Fig. 5 chapter](fig5.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
from glob import glob
import os
import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
from scipy.stats import zscore

from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.preprocessing import normalize

mpl.style.use("default")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'
dmrdir = f'{indir}analysis/DMR/'
loopdir = f'{indir}loop/majortype/'
outdir = f'{indir}analysis/loop_dmr_motif/'


In [ ]:
# ctx_db_fname = f'{dmrdir}merged_dmr_db/merged_dmr_slop1kb.regions_vs_motifs.rankings.feather'
# dem_db_fname = f'{dmrdir}merged_dmr_db/merged_dmr_slop1kb.regions_vs_motifs.scores.feather'


In [ ]:
# score = pd.read_feather(dem_db_fname)
# score

In [ ]:
# rank = pd.read_feather(ctx_db_fname)
# rank


In [ ]:
# import time
# from scipy.stats import rankdata

# start_time = time.time()
# for i in range(37):
#     if i==7:
#         continue
#     ct = f'c{i}'
#     print(ct)
#     os.makedirs(f'{outdir}{ct}/', exist_ok=True)
#     cmd = f'bedtools intersect -wa -a {dmrdir}merged_dmr.bed -b {dmrdir}overlap_atac/majortype_36groups/{ct}_merged.bed > {outdir}{ct}/dmr.bed'
#     os.system(cmd)
#     tmp = pd.read_csv(f'{outdir}{ct}/dmr.bed', sep='\t', names=['chrom','start','end'])
#     seldmr = tmp['chrom'].astype(str) + ':' + tmp['start'].astype(str) + '-' + tmp['end'].astype(str)
#     seldmr = score.columns.isin(seldmr)
#     # seldmr[-1] = True
#     score_tmp = score.loc[:, seldmr]
#     rank_tmp = rank.loc[:, seldmr]
#     print('Filter DMR', time.time()-start_time)
#     if i>0:
#         score_tmp['motifs'] = score['motifs']
#         score_tmp.to_feather(f'{outdir}{ct}/regions_vs_motifs.scores.feather')
#         print('Write score', time.time()-start_time)
#     rank_tmp = rankdata(rank_tmp, axis=1)
#     rank_tmp = pd.DataFrame(rank_tmp, index=rank.index, columns=score_tmp.columns)
#     print('Compute rank', time.time()-start_time)
#     rank_tmp['motifs'] = rank['motifs']
#     rank_tmp.to_feather(f'{outdir}{ct}/regions_vs_motifs.rankings.feather')
#     print('Write rank', time.time()-start_time)
    

In [ ]:
## DMR given size slop1k createdb
ct=c0
REGION_BED=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR/overlap_atac/majortype_36groups/${ct}_merged.bed
GENOME_FASTA=/large_storage/zhoulab/ref/hg38/fasta/hg38.fa
CHROMSIZES=/large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes
DATABASE_PREFIX=merge_dmr
SCRIPT_DIR=/home/zhoujt/software/create_cisTarget_databases
OUT_DIR=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif/${ct}
mkdir $OUT_DIR
${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
        ${GENOME_FASTA} \
        ${CHROMSIZES} \
        ${REGION_BED} \
        ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa \
        1000 \
        yes

export PATH=$PATH:~/software/
REF_DIR=/large_storage/zhoulab/ref/aertslab_motif_colleciton
CBDIR=${REF_DIR}/v10nr_clust_public/singletons
FASTA_FILE=${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa
MOTIF_LIST=${REF_DIR}/motifs.txt

${SCRIPT_DIR}/create_cistarget_motif_databases.py \
    -f ${FASTA_FILE} \
    -M ${CBDIR} \
    -m ${MOTIF_LIST} \
    -o ${OUT_DIR}/${DATABASE_PREFIX} \
    --bgpadding 1000 \
    -t 32


In [ ]:
## DMR 1kb createdb
ct=c0
REGION_BED=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR/overlap_atac/majortype_36groups/${ct}_merged.bed
GENOME_FASTA=/large_storage/zhoulab/ref/hg38/fasta/hg38.fa
CHROMSIZES=/large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes
DATABASE_PREFIX=merge_dmr
SCRIPT_DIR=/home/zhoujt/software/create_cisTarget_databases
OUT_DIR=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif/${ct}
mkdir $OUT_DIR
awk '{printf("%s\t%d\t%d\n",$1,($2+$3)/2,($2+$3)/2)}' $REGION_BED | bedtools slop -i - -b 500 -g /large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes > ${OUT_DIR}/${DATABASE_PREFIX}.1kb.bed
${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
        ${GENOME_FASTA} \
        ${CHROMSIZES} \
        ${OUT_DIR}/${DATABASE_PREFIX}.1kb.bed \
        ${OUT_DIR}/${DATABASE_PREFIX}.1kb.fa \
        0 \
        yes

export PATH=$PATH:~/software/
REF_DIR=/large_storage/zhoulab/ref/aertslab_motif_colleciton
CBDIR=${REF_DIR}/v10nr_clust_public/singletons
FASTA_FILE=${OUT_DIR}/${DATABASE_PREFIX}.1kb.fa
MOTIF_LIST=${REF_DIR}/motifs.txt

${SCRIPT_DIR}/create_cistarget_motif_databases.py \
    -f ${FASTA_FILE} \
    -M ${CBDIR} \
    -m ${MOTIF_LIST} \
    -o ${OUT_DIR}/${DATABASE_PREFIX}.1kb \
    --bgpadding 0 \
    -t 32


In [ ]:
## summit/loop DMR enriched motif

for i in 1 3; do (
ct=c${i}
loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
dmrdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR
outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif
mkdir -p ${outdir}/${ct}/bed/loop_dmr/ ${outdir}/${ct}/bed/summit_dmr/
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed

scenicplus grn_inference motif_enrichment_cistarget \
    --region_set_folder ${outdir}/${ct}/bed \
    --cistarget_db_fname ${outdir}/${ct}/merge_dmr.1kb.regions_vs_motifs.rankings.feather \
    --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.1kb.hdf5 \
    --temp_dir ${outdir}/${ct}/tmp \
    --species "homo_sapiens" \
    --fr_overlap_w_ctx_db 0.4 \
    --auc_threshold 0.005 \
    --nes_threshold 3.0 \
    --rank_threshold 0.05 \
    --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
    --annotation_version "v10nr_clust" \
    --motif_similarity_fdr 0.001 \
    --orthologous_identity_threshold 0.0 \
    --annotations_to_use "Direct_annot Orthology_annot" \
    --write_html \
    --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.1kb.html \
    --n_cpu 32

# for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.1kb.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; echo; done | cut -f6 | sort -k1,1 -u | less
) done


In [ ]:
## summit/loop DMR peak enriched motif

for i in `seq 35 36`; do (
ct=c${i}
loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
dmrdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR
outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif
mkdir -p ${outdir}/${ct}/bed/loop_dmr/ ${outdir}/${ct}/bed/summit_dmr/
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed

scenicplus grn_inference motif_enrichment_cistarget \
    --region_set_folder ${outdir}/${ct}/bed \
    --cistarget_db_fname ${outdir}/${ct}/merge_dmr.1kb.regions_vs_motifs.rankings.feather \
    --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.1kb.hdf5 \
    --temp_dir ${outdir}/${ct}/tmp \
    --species "homo_sapiens" \
    --fr_overlap_w_ctx_db 0.4 \
    --auc_threshold 0.005 \
    --nes_threshold 3.0 \
    --rank_threshold 0.05 \
    --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
    --annotation_version "v10nr_clust" \
    --motif_similarity_fdr 0.001 \
    --orthologous_identity_threshold 0.0 \
    --annotations_to_use "Direct_annot Orthology_annot" \
    --write_html \
    --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.1kb.html \
    --n_cpu 32

# for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.1kb.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; echo; done | cut -f6 | sort -k1,1 -u | less
) done


In [ ]:
for i in `seq 8 34`; do (
ct=c${i}
loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
dmrdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR
outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif
mkdir -p ${outdir}/${ct}/bed/loop_dmr/ ${outdir}/${ct}/bed/summit_dmr/
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed

scenicplus grn_inference motif_enrichment_cistarget \
    --region_set_folder ${outdir}/${ct}/bed \
    --cistarget_db_fname ${outdir}/${ct}/merge_dmr.regions_vs_motifs.rankings.feather \
    --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
    --temp_dir ${outdir}/${ct}/tmp \
    --species "homo_sapiens" \
    --fr_overlap_w_ctx_db 0.4 \
    --auc_threshold 0.005 \
    --nes_threshold 3.0 \
    --rank_threshold 0.05 \
    --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
    --annotation_version "v10nr_clust" \
    --motif_similarity_fdr 0.001 \
    --orthologous_identity_threshold 0.0 \
    --annotations_to_use "Direct_annot Orthology_annot" \
    --write_html \
    --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
    --n_cpu 32

# for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; done | cut -f6 | sort -k1,1 -u | less
) done


In [ ]:
## ATAC 1kb createdb

ct=c0
REGION_BED=/large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype/${ct}.bed
GENOME_FASTA=/large_storage/zhoulab/ref/hg38/fasta/hg38.fa
CHROMSIZES=/large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes
DATABASE_PREFIX=merge_peak
SCRIPT_DIR=/home/zhoujt/software/create_cisTarget_databases
OUT_DIR=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif/${ct}
mkdir $OUT_DIR
awk '{printf("%s\t%d\t%d\n",$1,($2+$3)/2,($2+$3)/2)}' $REGION_BED | bedtools slop -i - -b 500 -g /large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes > ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.bed
${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
        ${GENOME_FASTA} \
        ${CHROMSIZES} \
        ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.bed \
        ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa \
        0 \
        yes

export PATH=$PATH:~/software/
REF_DIR=/large_storage/zhoulab/ref/aertslab_motif_colleciton
CBDIR=${REF_DIR}/v10nr_clust_public/singletons
FASTA_FILE=${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa
MOTIF_LIST=${REF_DIR}/motifs.txt

"${SCRIPT_DIR}/create_cistarget_motif_databases.py" \
    -f ${FASTA_FILE} \
    -M ${CBDIR} \
    -m ${MOTIF_LIST} \
    -o ${OUT_DIR}/${DATABASE_PREFIX} \
    --bgpadding 0 \
    -t 32



In [ ]:
## summit/loop ATAC peak enriched motif

for i in `seq 3 34`; do (
ct=c${i}
loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
atacdir=/large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype
outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif
mkdir -p ${outdir}/${ct}/bed/loop_peak/ ${outdir}/${ct}/bed/summit_peak/
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_peak/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_peak/${ct}.bed

scenicplus grn_inference motif_enrichment_cistarget \
    --region_set_folder ${outdir}/${ct}/bed \
    --cistarget_db_fname ${outdir}/${ct}/merge_peak.regions_vs_motifs.rankings.feather \
    --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
    --temp_dir ${outdir}/${ct}/tmp \
    --species "homo_sapiens" \
    --fr_overlap_w_ctx_db 0.4 \
    --auc_threshold 0.005 \
    --nes_threshold 3.0 \
    --rank_threshold 0.05 \
    --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
    --annotation_version "v10nr_clust" \
    --motif_similarity_fdr 0.001 \
    --orthologous_identity_threshold 0.0 \
    --annotations_to_use "Direct_annot Orthology_annot" \
    --write_html \
    --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
    --n_cpu 32
)
done

for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; done | cut -f6 | sort -k1,1 -u | less


In [ ]:
ct=c35
loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
atacdir=/large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype
outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif
mkdir -p ${outdir}/${ct}/bed/loop_peak/ ${outdir}/${ct}/bed/summit_peak/
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_peak/${ct}.bed
awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_peak/${ct}.bed

scenicplus grn_inference motif_enrichment_cistarget \
    --region_set_folder ${outdir}/${ct}/bed \
    --cistarget_db_fname ${outdir}/${ct}/merge_peak.regions_vs_motifs.rankings.feather \
    --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
    --temp_dir ${outdir}/${ct}/tmp \
    --species "homo_sapiens" \
    --fr_overlap_w_ctx_db 0.4 \
    --auc_threshold 0.005 \
    --nes_threshold 3.0 \
    --rank_threshold 0.05 \
    --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
    --annotation_version "v10nr_clust" \
    --motif_similarity_fdr 0.001 \
    --orthologous_identity_threshold 0.0 \
    --annotations_to_use "Direct_annot Orthology_annot" \
    --write_html \
    --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
    --n_cpu 32
)
done
